## 대화 히스토리 압축 및 컨텍스트 관리

멀티턴 대화·RAG 에이전트는 매 턴마다 **시스템 프롬프트 + 과거 대화 + (검색 문서) + 현재 질문**을 LLM에 넣는다.  
대화가 길어지면 토큰이 컨텍스트 창을 잠식하고, 비용·지연·중간 정보 유실("lost in the middle")이 커진다.

| 전략 | 핵심 | 장점 | 단점 |
|---|---|---|---|
| **Sliding Window** | 최근 $N$턴만 유지 | 단순·빠름 | 오래된 사실(이름·결정) 소실 |
| **Token Budget Trim** | 토큰 상한까지 뒤에서부터 유지 | 창 크기와 정합 | 여전히 오래된 맥락 삭제 |
| **Summarization** | 과거 대화를 LLM으로 요약 | 장기 맥락 보존 | 요약 비용·정보 왜곡 가능 |
| **Hybrid** | 오래된 구간 요약 + 최근 원문 | 실무에서 가장 흔함 | 구현·임계값 튜닝 필요 |

```text
pip install langchain-openai langchain-core tiktoken python-dotenv
```

#### 기술 문서: 컨텍스트 예산(Context Budget)

한 번의 LLM 호출에 들어가는 입력은 대략 다음으로 나뉜다.

```text
총 입력 토큰 ≈ system + 대화히스토리 + RAG문서 + 현재질문 + (여유/출력예약)
```

- **히스토리만** 무제한으로 쌓으면 RAG 문서 자리가 줄어든다.
- **검색 문서만** 많이 넣으면 이전 대화(사용자 선호도·제약)를 잊는다.
- 실무에서는 **히스토리 예산**과 **RAG 예산**을 분리해 관리하는 것이 안전하다.

본 노트북은 `05_대화 메모리 관리`의 "저장" 위에, **무엇을 LLM에 넣을지 고르는 압축/트리밍**을 다룬다.

In [1]:
import os
from dotenv import load_dotenv

# .env 파일의 내용 불러오기
# OPENAI_API_KEY 필수
# 경로가 다르면 본인 환경에 맞게 수정한다.
load_dotenv("C:/env/.env")


True

### [0] 공통 준비: LLM, 토큰 카운터, 샘플 히스토리

#### 기술 문서: 왜 토큰을 세야 하는가

문자 수·메시지 개수는 컨텍스트 한도와 1:1이 아니다.  
OpenAI 계열은 보통 `tiktoken`의 `cl100k_base` / `o200k_base` 등으로 토큰화한다.  
압축 전략의 트리거는 **메시지 수**보다 **토큰 수**가 더 신뢰할 만하다.

| 구성요소 | 역할 | 본 노트북 선택 |
|---|---|---|
| **LLM** | 요약·최종 답변 | `gpt-4o-mini`, `temperature=0` |
| **tiktoken** | 입력 토큰 추정 | `cl100k_base` (gpt-4o-mini 근사) |
| **메시지** | Human / AI / System | LangChain `BaseMessage` |

In [3]:
from typing import List, Tuple

import tiktoken
from langchain_openai import ChatOpenAI
from langchain_core.messages import (
    AIMessage,
    BaseMessage,
    HumanMessage,
    SystemMessage,
)

# temperature=0: 전략 비교 시 출력을 재현 가능하게 유지
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# gpt-4o-mini 계열 토큰 추정용 인코딩
ENC = tiktoken.get_encoding("cl100k_base")


def count_tokens(text: str) -> int:
    """문자열 토큰 수 추정."""
    # encode 후 길이가 곧 토큰 개수
    return len(ENC.encode(text))


def count_message_tokens(messages: List[BaseMessage]) -> int:
    """메시지 리스트의 대략적 토큰 수.

    역할 토큰·특수 토큰은 모델마다 다르므로 +4/메시지 정도의 여유를 둔다.
    정확한 billing 토큰이 아니라 예산 관리용 추정값이다.
    """
    total = 0
    for m in messages:
        # content가 문자열이 아닐 수 있으므로 str로 방어
        total += count_tokens(m.content if isinstance(m.content, str) else str(m.content))
        total += 4  # role / framing overhead 근사
    return total


def print_messages(messages: List[BaseMessage], title: str = "messages") -> None:
    """메시지를 사람이 읽기 쉽게 출력하고 토큰 요약을 붙인다."""
    print(f"\n=== {title} (n={len(messages)}, ~{count_message_tokens(messages)} tokens) ===")
    for i, m in enumerate(messages):
        # 역할별 표시용 아이콘 (없으면 type 문자열 그대로)
        role = {"human": "🧑", "ai": "🤖", "system": "⚙️"}.get(m.type, m.type)
        content = m.content if isinstance(m.content, str) else str(m.content)
        # 긴 본문은 미리보기만 출력
        preview = content if len(content) <= 180 else content[:180] + "..."
        print(f"[{i}] {role} ({m.type}, {count_tokens(content)} tok): {preview}")


# 긴 멀티턴 샘플: 이름·선호·프로젝트 결정이 앞쪽에 있고, 뒤는 세부 논의
# → Sliding Window로 자르면 앞쪽 사실이 사라져 "까먹는" 현상을 재현하기 위함
SAMPLE_HISTORY: List[BaseMessage] = [
    HumanMessage(content="안녕. 나는 김민수야. RAG 챗봇을 만들고 있어."),
    AIMessage(content="안녕하세요 김민수님. RAG 챗봇 구성을 도와드릴게요. 어떤 단계부터 볼까요?"),
    HumanMessage(content="회사 내부 PDF를 검색해야 해. 답변은 한국어로, 근거 문장을 꼭 붙여줘."),
    AIMessage(content="알겠습니다. 한국어 답변 + 근거 인용을 기본 규칙으로 두겠습니다."),
    HumanMessage(content="임베딩은 text-embedding-3-small, 벡터DB는 FAISS로 가자."),
    AIMessage(content="좋아요. embedding-3-small + FAISS 조합으로 진행하겠습니다."),
    HumanMessage(content="청크 크기는 500, 오버랩 50이 괜찮을까?"),
    AIMessage(content="일반 PDF에는 합리적인 시작점입니다. 표·수식이 많으면 나중에 조정하세요."),
    HumanMessage(content="리랭커는 Cross-Encoder와 Cohere 중 뭐가 나아?"),
    AIMessage(content="로컬 비용 절감이면 Cross-Encoder, 운영 편의면 Cohere Rerank가 유리합니다."),
    HumanMessage(content="우선 로컬 Cross-Encoder로 가자. top_k=8, top_n=3."),
    AIMessage(content="확인했습니다. base k=8 → rerank top_n=3 파이프라인으로 고정할게요."),
    HumanMessage(content="프롬프트에 '문서에 없으면 모른다'고 명시해야지."),
    AIMessage(content="grounded generation 규칙을 시스템 프롬프트에 넣겠습니다."),
    HumanMessage(content="평가 지표는 Recall@k랑 MRR 위주로 볼게."),
    AIMessage(content="검색 품질은 Recall@k·MRR, 생성은 groundedness를 함께 보시면 좋습니다."),
    HumanMessage(content="야간 배치로 인덱스를 갱신할 예정이야."),
    AIMessage(content="배치 갱신 시 임베딩 모델 버전을 메타데이터에 남겨 드리프트를 추적하세요."),
    HumanMessage(content="사용자 피드백 버튼(👍/👎)도 넣을까?"),
    AIMessage(content="네. 쿼리·검색결과·답변을 함께 로깅하면 리랭크/청크 튜닝에 도움이 됩니다."),
]

# 모든 전략 실험에서 공통으로 쓰는 시스템 프롬프트
SYSTEM = SystemMessage(
    content=(
        "당신은 RAG 챗봇 설계 도우미다. "
        "이전 대화에서 확정된 사용자 이름·제약·기술 선택을 우선 반영하라."
    )
)

print_messages(SAMPLE_HISTORY, "원본 샘플 히스토리")
print(f"\nsystem tokens: {count_tokens(SYSTEM.content)}")



=== 원본 샘플 히스토리 (n=20, ~700 tokens) ===
[0] 🧑 (human, 30 tok): 안녕. 나는 김민수야. RAG 챗봇을 만들고 있어.
[1] 🤖 (ai, 51 tok): 안녕하세요 김민수님. RAG 챗봇 구성을 도와드릴게요. 어떤 단계부터 볼까요?
[2] 🧑 (human, 39 tok): 회사 내부 PDF를 검색해야 해. 답변은 한국어로, 근거 문장을 꼭 붙여줘.
[3] 🤖 (ai, 37 tok): 알겠습니다. 한국어 답변 + 근거 인용을 기본 규칙으로 두겠습니다.
[4] 🧑 (human, 24 tok): 임베딩은 text-embedding-3-small, 벡터DB는 FAISS로 가자.
[5] 🤖 (ai, 26 tok): 좋아요. embedding-3-small + FAISS 조합으로 진행하겠습니다.
[6] 🧑 (human, 26 tok): 청크 크기는 500, 오버랩 50이 괜찮을까?
[7] 🤖 (ai, 33 tok): 일반 PDF에는 합리적인 시작점입니다. 표·수식이 많으면 나중에 조정하세요.
[8] 🧑 (human, 20 tok): 리랭커는 Cross-Encoder와 Cohere 중 뭐가 나아?
[9] 🤖 (ai, 36 tok): 로컬 비용 절감이면 Cross-Encoder, 운영 편의면 Cohere Rerank가 유리합니다.
[10] 🧑 (human, 24 tok): 우선 로컬 Cross-Encoder로 가자. top_k=8, top_n=3.
[11] 🤖 (ai, 31 tok): 확인했습니다. base k=8 → rerank top_n=3 파이프라인으로 고정할게요.
[12] 🧑 (human, 26 tok): 프롬프트에 '문서에 없으면 모른다'고 명시해야지.
[13] 🤖 (ai, 27 tok): grounded generation 규칙을 시스템 프롬프트에 넣겠습니다.
[14] 🧑 (human, 22 tok): 평가 지표는 Recall@k랑 MRR 위주로 볼게.
[15] 🤖 (ai, 32 tok): 검색 품질은 Recall@k

### [1] Sliding Window: 최근 $N$턴만 유지

#### 기술 문서

가장 단순한 압축이다. Human–AI 쌍을 1턴으로 보고 **최근 `max_turns`만** 남긴다.

- **장점**: 구현이 쉽고 지연이 없다(추가 LLM 호출 없음).
- **위험**: 초반에 나온 이름·제약·기술 스택이 창 밖으로 밀려나면 모델이 "까먹는다".
- **적합**: 짧은 FAQ 봇, 턴당 독립성이 큰 업무.

In [4]:
def sliding_window(messages: List[BaseMessage], max_turns: int = 3) -> List[BaseMessage]:
    """최근 max_turns개의 (human, ai) 페어만 남긴다.

    메시지가 홀수 개(마지막 human만 있는 경우)여도 뒤에서부터 자른다.
    """
    # 1턴 = human + ai 이므로 메시지 수는 턴 수의 2배
    max_messages = max_turns * 2
    if len(messages) <= max_messages:
        return list(messages)  # 이미 창 안이면 복사만
    # 뒤에서부터 최근 메시지만 슬라이스
    return list(messages[-max_messages:])


# 최근 3턴(메시지 6개)만 남김 → 초반 이름/스택 정보는 탈락
windowed = sliding_window(SAMPLE_HISTORY, max_turns=3)
print_messages(windowed, "Sliding Window (최근 3턴)")

# 초반 정보가 빠졌는지 확인용 질문
probe = HumanMessage(content="내 이름이 뭐고, 벡터DB랑 리랭커 설정을 뭐로 하기로 했지?")
# system + 윈도우 히스토리 + 현재 질문
resp_window = llm.invoke([SYSTEM] + windowed + [probe])
print("\n[Sliding Window 답변]")
print(resp_window.content)



=== Sliding Window (최근 3턴) (n=6, ~214 tokens) ===
[0] 🧑 (human, 22 tok): 평가 지표는 Recall@k랑 MRR 위주로 볼게.
[1] 🤖 (ai, 32 tok): 검색 품질은 Recall@k·MRR, 생성은 groundedness를 함께 보시면 좋습니다.
[2] 🧑 (human, 19 tok): 야간 배치로 인덱스를 갱신할 예정이야.
[3] 🤖 (ai, 40 tok): 배치 갱신 시 임베딩 모델 버전을 메타데이터에 남겨 드리프트를 추적하세요.
[4] 🧑 (human, 27 tok): 사용자 피드백 버튼(👍/👎)도 넣을까?
[5] 🤖 (ai, 50 tok): 네. 쿼리·검색결과·답변을 함께 로깅하면 리랭크/청크 튜닝에 도움이 됩니다.

[Sliding Window 답변]
사용자 이름은 아직 언급되지 않았고, 벡터 DB는 Pinecone, 리랭커는 LightGBM으로 설정하기로 했습니다.


### [2] Token Budget Trim: 토큰 상한으로 자르기

#### 기술 문서

턴 수가 아니라 **토큰 예산**을 기준으로 한다. LangChain의 `trim_messages`는

- `strategy="last"`: 뒤에서부터 채움 (최근 대화 우선)
- `start_on="human"`: human으로 시작하는 유효한 대화를 유지
- `include_system=True`: 시스템 메시지는 보존

모델 컨텍스트 한도에서 **출력 예약 토큰**을 빼 둔 값을 `max_tokens`로 쓰는 것이 안전하다.

In [5]:
from langchain_core.messages import trim_messages
from langchain_core.messages.utils import count_tokens_approximately


def token_budget_trim(
    messages: List[BaseMessage],
    system: SystemMessage,
    max_tokens: int = 350,
) -> List[BaseMessage]:
    """시스템 + 히스토리를 토큰 예산 안으로 트리밍.

    trim_messages는 system을 포함해 예산을 계산할 수 있다.
    """
    return trim_messages(
        [system] + list(messages),  # system을 맨 앞에 두고 예산에 포함
        max_tokens=max_tokens,  # 입력 토큰 상한 (출력 예약분은 별도 확보 권장)
        strategy="last",  # 최근 메시지 우선으로 채움
        token_counter=count_tokens_approximately,  # LangChain 근사 카운터
        start_on="human",  # 잘린 결과가 human으로 시작하도록 맞춤
        include_system=True,  # system 메시지는 가급적 보존
        allow_partial=False,  # 메시지 중간 절단 금지 (통째로 포함/제외)
    )


# max_tokens를 작게 두면 앞쪽 대화가 잘리는지 관찰
trimmed = token_budget_trim(SAMPLE_HISTORY, SYSTEM, max_tokens=350)
print_messages(trimmed, "Token Budget Trim (max_tokens=350)")

probe = HumanMessage(content="내 이름이 뭐고, 임베딩 모델은 뭐로 정했지?")
# trimmed 안에 system이 이미 포함되어 있음
resp_trim = llm.invoke(trimmed + [probe])
print("\n[Token Trim 답변]")
print(resp_trim.content)



=== Token Budget Trim (max_tokens=350) (n=21, ~757 tokens) ===
[0] ⚙️ (system, 53 tok): 당신은 RAG 챗봇 설계 도우미다. 이전 대화에서 확정된 사용자 이름·제약·기술 선택을 우선 반영하라.
[1] 🧑 (human, 30 tok): 안녕. 나는 김민수야. RAG 챗봇을 만들고 있어.
[2] 🤖 (ai, 51 tok): 안녕하세요 김민수님. RAG 챗봇 구성을 도와드릴게요. 어떤 단계부터 볼까요?
[3] 🧑 (human, 39 tok): 회사 내부 PDF를 검색해야 해. 답변은 한국어로, 근거 문장을 꼭 붙여줘.
[4] 🤖 (ai, 37 tok): 알겠습니다. 한국어 답변 + 근거 인용을 기본 규칙으로 두겠습니다.
[5] 🧑 (human, 24 tok): 임베딩은 text-embedding-3-small, 벡터DB는 FAISS로 가자.
[6] 🤖 (ai, 26 tok): 좋아요. embedding-3-small + FAISS 조합으로 진행하겠습니다.
[7] 🧑 (human, 26 tok): 청크 크기는 500, 오버랩 50이 괜찮을까?
[8] 🤖 (ai, 33 tok): 일반 PDF에는 합리적인 시작점입니다. 표·수식이 많으면 나중에 조정하세요.
[9] 🧑 (human, 20 tok): 리랭커는 Cross-Encoder와 Cohere 중 뭐가 나아?
[10] 🤖 (ai, 36 tok): 로컬 비용 절감이면 Cross-Encoder, 운영 편의면 Cohere Rerank가 유리합니다.
[11] 🧑 (human, 24 tok): 우선 로컬 Cross-Encoder로 가자. top_k=8, top_n=3.
[12] 🤖 (ai, 31 tok): 확인했습니다. base k=8 → rerank top_n=3 파이프라인으로 고정할게요.
[13] 🧑 (human, 26 tok): 프롬프트에 '문서에 없으면 모른다'고 명시해야지.
[14] 🤖 (ai, 27 tok): grounded generation 규칙

### [3] Summarization: 과거 대화를 요약으로 압축

#### 기술 문서

삭제 대신 **정보 밀도를 높인다**. 오래된 메시지를 LLM에 넣어 짧은 요약(사실·결정·제약)으로 바꾼다.

```text
[오래된 N개 메시지] --(요약 LLM)--> [요약 System/AI 메시지] + [최근 원문]
```

- **장점**: 이름·선호·확정 스택 같은 long-range 사실을 남기기 쉽다.
- **단점**: 요약 호출 비용, 요약 품질에 따른 환각/누락.
- **팁**: 요약 프롬프트에 "결정사항·제약·고유명사·숫자"를 우선 보존하라고 명시한다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 요약 품질을 위해 "무엇을 남길지"를 프롬프트에 명시
summarize_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "다음 대화를 한국어로 간결히 요약하라.\n"
            "반드시 포함할 것: 사용자 이름, 확정된 기술 선택, 제약/규칙, 숫자 설정.\n"
            "추측하지 말고 대화에 있는 사실만 적어라. 불릿 3~8개.",
        ),
        ("human", "{dialogue}"),
    ]
)
# LCEL: prompt → llm → 문자열 파서
summarize_chain = summarize_prompt | llm | StrOutputParser()


def format_dialogue(messages: List[BaseMessage]) -> str:
    """메시지 리스트를 요약 LLM에 넣기 쉬운 대화 텍스트로 변환."""
    lines = []
    for m in messages:
        role = {"human": "User", "ai": "Assistant", "system": "System"}.get(m.type, m.type)
        lines.append(f"{role}: {m.content}")
    return "\n".join(lines)


def summarize_messages(messages: List[BaseMessage]) -> str:
    """메시지 구간을 사실 중심 요약 문자열로 압축."""
    return summarize_chain.invoke({"dialogue": format_dialogue(messages)})


# 전체 히스토리를 한 번에 요약 (비교용)
full_summary = summarize_messages(SAMPLE_HISTORY)
print("=== 전체 히스토리 요약 ===")
print(full_summary)
print(f"\n원본 ~{count_message_tokens(SAMPLE_HISTORY)} tok → 요약 {count_tokens(full_summary)} tok")

# 요약문을 별도 system 메시지로 넣어 원문 히스토리 대신 사용
summary_as_system = SystemMessage(
    content="[이전 대화 요약]\n" + full_summary
)
probe = HumanMessage(content="내 이름, 벡터DB, 리랭커 top_k/top_n을 말해줘.")
resp_sum = llm.invoke([SYSTEM, summary_as_system, probe])
print("\n[Summarization만 사용 답변]")
print(resp_sum.content)


### [4] Hybrid: 오래된 구간 요약 + 최근 원문

#### 기술 문서: 실무 기본 패턴

```text
if total_tokens > threshold:
    old, recent = split(history, keep_recent_turns)
    summary = summarize(old)          # 또는 누적 running summary 갱신
    context = [system, summary, *recent, current_user]
else:
    context = [system, *history, current_user]
```

- **recent 원문**: 직전 참조("그것", "방금 말한")에 강하다.
- **old 요약**: 초반 결정사항을 값싸게 유지한다.
- **running summary**: 매 턴 전체를 다시 요약하지 말고, `기존요약 + 새 구간`만 점진 갱신하면 비용이 안정된다.

In [6]:
def hybrid_compress(
    messages: List[BaseMessage],
    keep_recent_turns: int = 3,
    token_threshold: int = 400,
) -> Tuple[List[BaseMessage], dict]:
    """토큰이 임계값을 넘으면 앞부분을 요약하고 최근 턴은 원문으로 유지.

    Returns:
        compressed messages (system 요약 메시지 포함 가능), meta dict
    """
    meta = {
        "original_tokens": count_message_tokens(messages),
        "compressed": False,
        "summary": None,
    }

    # 임계값 이하면 압축하지 않고 원문 그대로
    if meta["original_tokens"] <= token_threshold:
        return list(messages), meta

    keep_n = keep_recent_turns * 2  # 최근 턴 → 메시지 개수
    if len(messages) <= keep_n:
        return list(messages), meta

    # 앞(old)=요약 대상, 뒤(recent)=원문 유지
    old, recent = messages[:-keep_n], messages[-keep_n:]
    summary = summarize_messages(old)
    meta.update({"compressed": True, "summary": summary})

    # 요약을 system 메시지로 앞에 두고 최근 원문을 이어 붙임
    compressed: List[BaseMessage] = [
        SystemMessage(content="[이전 대화 요약]\n" + summary),
        *recent,
    ]
    meta["compressed_tokens"] = count_message_tokens(compressed)
    return compressed, meta


# token_threshold=400: 샘플(~700tok)이 임계를 넘겨 압축이 실행됨
hybrid, hybrid_meta = hybrid_compress(
    SAMPLE_HISTORY, keep_recent_turns=3, token_threshold=400
)
# summary 본문은 길어서 meta에서 제외하고 따로 출력
print("meta:", {k: v for k, v in hybrid_meta.items() if k != "summary"})
if hybrid_meta.get("summary"):
    print("\n[요약]")
    print(hybrid_meta["summary"])
print_messages(hybrid, "Hybrid (요약 + 최근 3턴)")

# 초반 사실(이름/스택) + 최근 사실(피드백)을 동시에 묻는다
probe = HumanMessage(
    content="내 이름과 FAISS/리랭커 설정, 그리고 방금 논의한 피드백 버튼 의견을 짧게 정리해줘."
)
resp_hybrid = llm.invoke([SYSTEM] + hybrid + [probe])
print("\n[Hybrid 답변]")
print(resp_hybrid.content)


meta: {'original_tokens': 700, 'compressed': True, 'compressed_tokens': 357}

[요약]
- 사용자 이름: 김민수
- 확정된 기술 선택: 임베딩은 text-embedding-3-small, 벡터DB는 FAISS, 리랭커는 Cross-Encoder
- 제약/규칙: 답변은 한국어로, 근거 문장 포함, '문서에 없으면 모른다' 명시
- 숫자 설정: 청크 크기 500, 오버랩 50, top_k=8, top_n=3

=== Hybrid (요약 + 최근 3턴) (n=7, ~357 tokens) ===
[0] ⚙️ (system, 139 tok): [이전 대화 요약]
- 사용자 이름: 김민수
- 확정된 기술 선택: 임베딩은 text-embedding-3-small, 벡터DB는 FAISS, 리랭커는 Cross-Encoder
- 제약/규칙: 답변은 한국어로, 근거 문장 포함, '문서에 없으면 모른다' 명시
- 숫자 설정: 청크 크기 500, 오버랩 50, top_k=8...
[1] 🧑 (human, 22 tok): 평가 지표는 Recall@k랑 MRR 위주로 볼게.
[2] 🤖 (ai, 32 tok): 검색 품질은 Recall@k·MRR, 생성은 groundedness를 함께 보시면 좋습니다.
[3] 🧑 (human, 19 tok): 야간 배치로 인덱스를 갱신할 예정이야.
[4] 🤖 (ai, 40 tok): 배치 갱신 시 임베딩 모델 버전을 메타데이터에 남겨 드리프트를 추적하세요.
[5] 🧑 (human, 27 tok): 사용자 피드백 버튼(👍/👎)도 넣을까?
[6] 🤖 (ai, 50 tok): 네. 쿼리·검색결과·답변을 함께 로깅하면 리랭크/청크 튜닝에 도움이 됩니다.

[Hybrid 답변]
김민수님은 FAISS를 벡터DB로 사용하고, Cross-Encoder를 리랭커로 설정했습니다. 또한, 사용자 피드백 버튼(👍/👎)을 추가하여 쿼리·검색결과·답변을 로깅할 계획입니다.


### [5] Running Summary: 점진적 요약 갱신

#### 기술 문서

매 요청마다 전체 히스토리를 다시 요약하면 비용이 $O(T^2)$에 가깝게 증가한다.  
**running summary**는 기존 요약과 "새로 추가된 구간"만 합쳐 갱신한다.

```text
summary_t = summarize(summary_{t-1} + new_messages)
```

세션 상태에는 보통 `running_summary` + `recent_raw_messages` 두 필드를 둔다.

In [7]:
# 기존 요약 + 새 구간만 받아 점진 갱신하는 프롬프트
running_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "기존 요약과 새 대화를 합쳐 갱신된 요약을 작성하라.\n"
            "고유명사·숫자·확정 결정·제약을 우선 보존하고, 중복은 제거하라.\n"
            "한국어 불릿 3~8개.",
        ),
        (
            "human",
            "[기존 요약]\n{summary}\n\n[새 대화]\n{new_dialogue}",
        ),
    ]
)
running_chain = running_prompt | llm | StrOutputParser()


class RunningSummaryMemory:
    """running summary + 최근 원문 버퍼."""

    def __init__(self, keep_recent_turns: int = 2, summarize_every_turns: int = 3):
        self.keep_recent_turns = keep_recent_turns  # LLM에 넣을 최근 원문 턴 수
        self.summarize_every_turns = summarize_every_turns  # 몇 턴마다 요약 갱신할지
        self.summary: str = "(없음)"  # 누적(running) 요약
        self.recent: List[BaseMessage] = []  # 최근 원문 버퍼
        self._pending: List[BaseMessage] = []  # 아직 요약에 반영되지 않은 구간
        self.turn_count = 0

    def add_exchange(self, human: str, ai: str) -> None:
        """한 턴(human+ai)을 버퍼에 추가하고, 주기에 맞춰 요약을 갱신."""
        self.recent.extend([HumanMessage(content=human), AIMessage(content=ai)])
        self._pending.extend([HumanMessage(content=human), AIMessage(content=ai)])
        self.turn_count += 1

        # 최근 원문 길이 제한
        max_recent = self.keep_recent_turns * 2
        if len(self.recent) > max_recent:
            overflow = self.recent[:-max_recent]
            self.recent = self.recent[-max_recent:]
            # overflow는 pending에 이미 포함되어 요약 때 반영

        # N턴마다 pending 구간을 running summary에 흡수
        if self.turn_count % self.summarize_every_turns == 0 and self._pending:
            self.summary = running_chain.invoke(
                {
                    "summary": self.summary,
                    "new_dialogue": format_dialogue(self._pending),
                }
            )
            self._pending = []  # 반영 완료 → pending 비움

    def build_context(self, system: SystemMessage) -> List[BaseMessage]:
        """LLM 호출용 메시지: system + (누적 요약) + 최근 원문."""
        msgs: List[BaseMessage] = [system]
        if self.summary and self.summary != "(없음)":
            msgs.append(SystemMessage(content="[누적 대화 요약]\n" + self.summary))
        msgs.extend(self.recent)
        return msgs


# 샘플 히스토리를 턴 단위로 재생하며 running summary 관찰
mem = RunningSummaryMemory(keep_recent_turns=2, summarize_every_turns=3)
# (human, ai) 페어로 재구성
pairs = [
    (SAMPLE_HISTORY[i].content, SAMPLE_HISTORY[i + 1].content)
    for i in range(0, len(SAMPLE_HISTORY), 2)
]

for t, (h, a) in enumerate(pairs, start=1):
    mem.add_exchange(h, a)
    # summarize_every_turns=3 이므로 3의 배수 턴에서 요약 스냅샷 출력
    if t % 3 == 0:
        print(f"\n--- turn {t}: running summary ---")
        print(mem.summary)

ctx = mem.build_context(SYSTEM)
print_messages(ctx, "Running Summary 최종 컨텍스트")

probe = HumanMessage(content="확정된 기술 스택과 평가 지표를 말해줘. 내 이름도.")
resp_run = llm.invoke(ctx + [probe])
print("\n[Running Summary 답변]")
print(resp_run.content)



--- turn 3: running summary ---
- 사용자 이름: 김민수
- RAG 챗봇 개발 중
- 회사 내부 PDF 검색 필요
- 답변은 한국어로 제공, 근거 문장 포함
- 임베딩 모델: text-embedding-3-small
- 벡터 데이터베이스: FAISS 사용

--- turn 6: running summary ---
- 사용자 이름: 김민수
- RAG 챗봇 개발 중
- 회사 내부 PDF 검색 필요
- 답변은 한국어로 제공, 근거 문장 포함
- 임베딩 모델: text-embedding-3-small
- 벡터 데이터베이스: FAISS 사용
- 청크 크기: 500, 오버랩: 50
- 리랭커: 로컬 Cross-Encoder 사용, top_k=8, top_n=3 설정

--- turn 9: running summary ---
- 사용자 이름: 김민수
- RAG 챗봇 개발 중
- 회사 내부 PDF 검색 필요
- 답변은 한국어로 제공, 근거 문장 포함
- 임베딩 모델: text-embedding-3-small
- 벡터 데이터베이스: FAISS 사용
- 청크 크기: 500, 오버랩: 50
- 리랭커: 로컬 Cross-Encoder 사용, top_k=8, top_n=3 설정
- 프롬프트에 '문서에 없으면 모른다' 명시 예정
- 평가 지표: Recall@k와 MRR 위주
- 야간 배치로 인덱스 갱신 예정, 임베딩 모델 버전 메타데이터에 기록 필요

=== Running Summary 최종 컨텍스트 (n=6, ~442 tokens) ===
[0] ⚙️ (system, 53 tok): 당신은 RAG 챗봇 설계 도우미다. 이전 대화에서 확정된 사용자 이름·제약·기술 선택을 우선 반영하라.
[1] ⚙️ (system, 229 tok): [누적 대화 요약]
- 사용자 이름: 김민수
- RAG 챗봇 개발 중
- 회사 내부 PDF 검색 필요
- 답변은 한국어로 제공, 근거 문장 포함
- 임베딩 모델: text-embedding-3-small
- 벡터 데이터베이스: F

### [6] 전략 비교: 같은 질문에 대한 토큰·정답 보존

아래는 **동일 probe 질문**에 대해 각 전략의 입력 토큰과 응답을 나란히 본다.  
(실행 시 API 비용이 발생한다.)

In [8]:
# 모든 전략에 동일한 probe를 넣어 토큰·정답 보존을 비교
probe = HumanMessage(
    content=(
        "다음을 빠짐없이 답해줘: "
        "(1) 내 이름 (2) 임베딩 모델 (3) 벡터DB (4) 리랭커와 top_k/top_n "
        "(5) 답변 언어/근거 규칙"
    )
)

strategies = {}

# 1) 원본 전체
full_ctx = [SYSTEM] + SAMPLE_HISTORY + [probe]
strategies["full"] = full_ctx

# 2) sliding window
strategies["sliding"] = [SYSTEM] + sliding_window(SAMPLE_HISTORY, max_turns=3) + [probe]

# 3) token trim
strategies["token_trim"] = token_budget_trim(SAMPLE_HISTORY, SYSTEM, max_tokens=350) + [probe]

# 4) hybrid
h_msgs, _ = hybrid_compress(SAMPLE_HISTORY, keep_recent_turns=3, token_threshold=400)
strategies["hybrid"] = [SYSTEM] + h_msgs + [probe]

# 5) running summary (이미 mem에 구축됨)
strategies["running"] = mem.build_context(SYSTEM) + [probe]

# 전략별 입력 토큰과 답변을 순서대로 호출·기록
rows = []
for name, ctx in strategies.items():
    tok = count_message_tokens(ctx)
    answer = llm.invoke(ctx).content
    rows.append((name, tok, answer))
    print(f"\n{'=' * 60}\n[{name}] input ~{tok} tokens\n{'-' * 60}\n{answer}")

print("\n=== 토큰 요약 ===")
for name, tok, _ in rows:
    print(f"{name:12s}: ~{tok} tokens")



[full] input ~835 tokens
------------------------------------------------------------
(1) 김민수  
(2) text-embedding-3-small  
(3) FAISS  
(4) Cross-Encoder와 top_k=8, top_n=3  
(5) 한국어 답변 + 근거 문장 포함

[sliding] input ~349 tokens
------------------------------------------------------------
(1) 사용자 이름: [사용자 이름]  
(2) 임베딩 모델: [임베딩 모델]  
(3) 벡터DB: [벡터DB]  
(4) 리랭커와 top_k/top_n: [리랭커]와 top_k/top_n 설정  
(5) 답변 언어/근거 규칙: [답변 언어] 및 [근거 규칙]  

각 항목에 대한 구체적인 내용을 추가해 주세요!

[token_trim] input ~835 tokens
------------------------------------------------------------
물론입니다! 다음은 요청하신 내용입니다:

(1) 이름: 김민수  
(2) 임베딩 모델: text-embedding-3-small  
(3) 벡터DB: FAISS  
(4) 리랭커: Cross-Encoder, top_k=8, top_n=3  
(5) 답변 언어: 한국어, 근거 문장 포함  

추가로 필요한 사항이 있으면 말씀해 주세요!

[hybrid] input ~493 tokens
------------------------------------------------------------
(1) 당신의 이름: 김민수  
(2) 임베딩 모델: text-embedding-3-small  
(3) 벡터DB: FAISS  
(4) 리랭커: Cross-Encoder, top_k: 8, top_n: 3  
(5) 답변 언어: 한국어, 근거 규칙: 문서에 없으면 모른다.

[running] 

### [7] RAG와 함께: 컨텍스트 예산 분할

#### 기술 문서

대화형 RAG에서는 히스토리 압축과 **검색 문서 예산**을 같이 설계해야 한다.

| 구간 | 권장 역할 | 예시 예산 비율 |
|---|---|---|
| System + 도구 규칙 | 고정 | 5~10% |
| 대화 히스토리(압축 후) | 선호도·이전 결정 | 20~30% |
| RAG 문서 | grounded 근거 | 40~50% |
| 현재 질문 + 출력 예약 | 질의·생성 | 15~20% |

검색 문서가 길면 히스토리와 동일한 아이디어를 적용할 수 있다.

- **top_n 축소** / 리랭크로 관련 문서만 남기기
- **문서 요약·추출** (Contextual Compression)
- **인용에 필요한 span만** 프롬프트에 넣기

아래는 "히스토리 예산 + RAG 예산"을 분리해 조립하는 스케치다.

In [9]:
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate


def fit_docs_to_budget(docs: List[Document], max_tokens: int) -> List[Document]:
    """문서를 앞에서부터 채워 토큰 예산에 맞춘다 (이미 리랭크된 순서를 가정)."""
    kept = []
    used = 0
    for d in docs:
        t = count_tokens(d.page_content) + 8  # 번호/구분자 overhead 근사
        if used + t > max_tokens:
            break  # 예산 초과 직전에서 중단 (뒤 문서는 버림)
        kept.append(d)
        used += t
    return kept


def build_rag_messages(
    system: SystemMessage,
    history: List[BaseMessage],
    docs: List[Document],
    question: str,
    history_budget: int = 500,
    docs_budget: int = 600,
) -> List[BaseMessage]:
    """히스토리·문서를 각각의 예산으로 압축한 뒤 최종 메시지 리스트를 만든다."""
    # 히스토리: hybrid 압축 후 필요 시 추가 트리밍
    compressed, _ = hybrid_compress(
        history, keep_recent_turns=2, token_threshold=history_budget
    )
    hist_msgs = token_budget_trim(compressed, system, max_tokens=history_budget)

    # 문서는 리랭크된 상위부터 예산에 맞게 채움
    kept_docs = fit_docs_to_budget(docs, max_tokens=docs_budget)
    docs_block = "\n\n".join(
        f"[{i+1}] {d.page_content}" for i, d in enumerate(kept_docs)
    )

    # hist_msgs에 system이 이미 포함되어 있으므로 문서/질문은 뒤에 추가
    return hist_msgs + [
        SystemMessage(content="[검색 문서]\n" + (docs_block or "(없음)")),
        HumanMessage(content=question),
    ]


# 가짜 검색 결과로 예산 조립 데모
# 마지막 문서는 의도적으로 길어 docs_budget을 넘기도록 패딩
fake_docs = [
    Document(page_content="FAISS는 로컬 밀집 벡터 검색에 널리 쓰이며 CPU 환경에서도 동작한다."),
    Document(page_content="Cross-Encoder 리랭커는 질문-문서 쌍을 함께 인코딩해 관련성 점수를 낸다."),
    Document(page_content="grounded generation은 제공된 문맥 밖의 지식을 사용하지 않도록 강제한다."),
    Document(page_content="이 문서는 예산 초과를 유도하기 위한 긴 패딩입니다. " * 40),
]

rag_msgs = build_rag_messages(
    SYSTEM,
    SAMPLE_HISTORY,
    fake_docs,
    question="우리 스택 기준으로 리랭크 단계를 한 문장으로 설명해줘. 문서 근거를 붙여.",
    history_budget=450,  # 대화 히스토리용 토큰 예산
    docs_budget=250,  # RAG 문서용 토큰 예산 (긴 패딩 문서는 탈락 예상)
)
print_messages(rag_msgs, "RAG용 조립 컨텍스트")

rag_answer = llm.invoke(rag_msgs)
print("\n[RAG + 압축 히스토리 답변]")
print(rag_answer.content)



=== RAG용 조립 컨텍스트 (n=7, ~376 tokens) ===
[0] ⚙️ (system, 53 tok): 당신은 RAG 챗봇 설계 도우미다. 이전 대화에서 확정된 사용자 이름·제약·기술 선택을 우선 반영하라.
[1] 🧑 (human, 19 tok): 야간 배치로 인덱스를 갱신할 예정이야.
[2] 🤖 (ai, 40 tok): 배치 갱신 시 임베딩 모델 버전을 메타데이터에 남겨 드리프트를 추적하세요.
[3] 🧑 (human, 27 tok): 사용자 피드백 버튼(👍/👎)도 넣을까?
[4] 🤖 (ai, 50 tok): 네. 쿼리·검색결과·답변을 함께 로깅하면 리랭크/청크 튜닝에 도움이 됩니다.
[5] ⚙️ (system, 120 tok): [검색 문서]
[1] FAISS는 로컬 밀집 벡터 검색에 널리 쓰이며 CPU 환경에서도 동작한다.

[2] Cross-Encoder 리랭커는 질문-문서 쌍을 함께 인코딩해 관련성 점수를 낸다.

[3] grounded generation은 제공된 문맥 밖의 지식을 사용하지 않도록 강제한다.
[6] 🧑 (human, 39 tok): 우리 스택 기준으로 리랭크 단계를 한 문장으로 설명해줘. 문서 근거를 붙여.

[RAG + 압축 히스토리 답변]
리랭크 단계는 Cross-Encoder를 사용하여 질문-문서 쌍을 함께 인코딩하고 관련성 점수를 산출하는 과정입니다(문서 [2]).


### [8] 실무 체크리스트 / 정리

| 상황 | 추천 |
|---|---|
| 짧은 세션·저비용 | Sliding Window 또는 Token Trim |
| 사용자 프로필·결정이 중요 | Hybrid 또는 Running Summary |
| 대화형 RAG | 히스토리 예산과 문서 예산을 **분리** |
| 초장문 세션 | Running Summary + 최근 $N$턴 원문 |

**설계 포인트**

1. **저장**과 **프롬프트 투입**을 분리한다. DB에는 원문을 두고, LLM에는 압축본만 넣는다.
2. 트리거는 메시지 개수보다 **토큰 임계값**이 안전하다.
3. 요약 프롬프트에 **고유명사·숫자·제약** 보존을 명시한다.
4. 요약본만으로 답이 흔들리면 `keep_recent_turns`를 늘리거나, 중요 슬롯(이름·스택)을 구조화 필드로 별도 저장한다.
5. `05_대화 메모리 관리`의 File/SQL/Redis 히스토리와 이 노트북의 압축기를 조합하면 운영형 멀티턴 RAG가 된다.

```text
저장소(원문) → (토큰 임계?) → Hybrid/Running Summary → [system+요약+최근+RAG docs+질문] → LLM
```